# 🔍 Trực quan hóa kết quả: Nhãn thực tế (Ground Truth) vs Dự đoán của mô hình (Predictions)
Notebook này giúp bạn đối chiếu trực quan kết quả nhận diện ổ gà của mô hình YOLOv8 trên tập dữ liệu Việt Nam.

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import pathlib
from ultralytics import YOLO

# Tắt cảnh báo verbose của YOLO
import logging
logging.getLogger("ultralytics").setLevel(logging.ERROR)

In [ ]:
def get_label_path(image_path):
    """Chuyển đổi đường dẫn ảnh thành đường dẫn nhãn YOLO tương ứng."""
    p = pathlib.Path(image_path)
    # Đường dẫn cha của thư mục images là .../test hoặc .../valid
    # Ví dụ: .../test/images/img.jpg -> .../test/labels/img.txt
    label_dir = p.parent.parent / 'labels'
    label_path = label_dir / (p.stem + '.txt')
    return str(label_path)

def plot_gt_vs_pred(image_path, model):
    """Vẽ nhãn thực tế (Ground Truth) và Dự đoán (Prediction) song song."""
    if not os.path.exists(image_path):
        print(f"❌ Không tìm thấy ảnh tại: {image_path}")
        return
        
    # 1. Đọc ảnh
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    img_gt = img.copy()
    img_pred = img.copy()
    
    # 2. Vẽ Nhãn thực tế (Ground Truth) màu XANH LÁ
    label_path = get_label_path(image_path)
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(parts[0])
                x_c, y_c, bw, bh = map(float, parts[1:5])
                # Chuyển đổi tọa độ chuẩn hóa sang tọa độ pixel
                x1 = int((x_c - bw/2) * w)
                y1 = int((y_c - bh/2) * h)
                x2 = int((x_c + bw/2) * w)
                y2 = int((y_c + bh/2) * h)
                cv2.rectangle(img_gt, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_gt, "Pothole (GT)", (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        cv2.putText(img_gt, "Khong tim thay file label!", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                    
    # 3. Vẽ BBox mô hình dự đoán màu ĐỎ
    results = model(image_path)[0]
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        conf = float(box.conf[0])
        cv2.rectangle(img_pred, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img_pred, f"Pothole {conf:.2f}", (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                    
    # 4. Hiển thị song song bằng Matplotlib
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(img_gt)
    axes[0].set_title("Ground Truth (Nhãn Thực Tế)", fontsize=14, color="green", fontweight="bold")
    axes[0].axis("off")
    
    axes[1].imshow(img_pred)
    axes[1].set_title("Prediction (Máy Dự Đoán)", fontsize=14, color="red", fontweight="bold")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 📁 Cấu hình Mô hình & Dữ liệu
Chọn mô hình weights và tập dữ liệu bạn muốn kiểm tra ở dưới:

In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules import block
from ultralytics.nn import tasks

# ==============================================================================
# SimAM DYNAMIC PATCHING (BẮT BUỘC ĐỂ LOAD TRỌNG SỐ SimAM)
# Chạy ô này TRƯỚC KHI load model YOLOv8+SimAM!
# ==============================================================================
class SimAM(nn.Module):
    def __init__(self, e_lambda=1e-4):
        super(SimAM, self).__init__()
        self.activation = nn.Sigmoid()
        self.e_lambda = e_lambda

    def forward(self, x):
        b, c, h, w = x.size()
        n = w * h - 1
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)) + 0.5
        return x * self.activation(y)

OriginalC2f = block.C2f

class C2f_SimAM(OriginalC2f):
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.simam = SimAM()
        
    def forward(self, x):
        out = super().forward(x)
        return self.simam(out)

print("🔧 Đang áp dụng SimAM monkey-patch vào kiến trúc YOLOv8...")
block.C2f = C2f_SimAM
tasks.C2f = C2f_SimAM


In [ ]:
import glob

# Xác định thư mục dự án (Tự động thích ứng dù chạy ở root hay thư mục notebooks)
current_dir = os.path.abspath("")
if current_dir.endswith("notebooks"):
    PROJECT_ROOT = os.path.dirname(current_dir)
else:
    PROJECT_ROOT = current_dir

# 1. Đường dẫn tới checkpoint tốt nhất của bạn
# Sửa thành đường dẫn SimAM mà bạn muốn:
model_path = os.path.join(PROJECT_ROOT, "runs", "vietnam_evaluation", "full_lpft", "yolo_simam", "fold_1", "ft", "weights", "best.pt")

print(f"📦 Đang load mô hình từ: {model_path}")
model = YOLO(model_path)

# 2. Đường dẫn tới thư mục ảnh test hoặc valid (VietNamPotholeDataset_External)
images_dir = os.path.join(PROJECT_ROOT, "data", "processed", "VietNamPotholeDataset_External", "test", "images")

# Lấy tất cả các file ảnh jpg và png trong thư mục test
image_paths = glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.png"))

print(f"✅ Tìm thấy {len(image_paths)} ảnh trong danh sách đánh giá.\n")


## 🎨 Chạy trực quan hóa ngẫu nhiên
*Chạy ô dưới đây nhiều lần (nhấn `Ctrl + Enter` hoặc nút Run) để lấy ngẫu nhiên ảnh trong tập test và so sánh.*

In [ ]:
# Lấy ngẫu nhiên 1 ảnh và vẽ kết quả
random_img = random.choice(image_paths)
print(f"📱 Đang xem ảnh: {os.path.basename(random_img)}")
plot_gt_vs_pred(random_img, model)